# Section 4. 간단한 Agent Workflow

Agent workflow는 한 번의 LLM 호출로 끝내지 않고, 검색, 판단, 답변, 검증 같은 단계를 연결하는 구조입니다.

여기서는 복잡한 프레임워크를 쓰지 않고 다음 흐름을 직접 만듭니다.

1. 질문을 받습니다.
2. 관련 문서를 검색합니다.
3. 근거가 있으면 답변합니다.
4. 근거가 없으면 사람 검토나 추가 자료 요청으로 보냅니다.


In [ ]:
CORPUS = [
    {
        "source_id": "support_triage",
        "text": "Support triage routes billing requests to the finance queue. Urgent security issues go to the incident channel. Uncertain cases use a fallback path for review.",
    },
    {
        "source_id": "api_onboarding",
        "text": "API onboarding requires authentication, environment variables, awareness of rate limits, and one minimal smoke test before larger experiments.",
    },
    {
        "source_id": "rag_quality",
        "text": "A grounded RAG answer should cite the retrieved source, quote only text that appears in that source, and say not answered when the corpus lacks evidence.",
    },
    {
        "source_id": "release_notes",
        "text": "Release notes summarize user-visible changes, known limitations, and migration steps. They are not a source of private pricing policy.",
    },
]


In [ ]:
import re
from pydantic import BaseModel, Field

class Evidence(BaseModel):
    source_id: str
    quote: str

class WorkflowResult(BaseModel):
    route: str = Field(description="answer, not_answered, needs_review 중 하나")
    answer_ko: str
    evidence: list[Evidence]
    next_action_ko: str

def tokenize(text: str) -> set[str]:
    return set(re.findall(r"[a-zA-Z0-9_]+", text.lower()))

def retrieve(query: str, top_k: int = 2) -> list[dict]:
    q = tokenize(query)
    scored = []
    for doc in CORPUS:
        score = len(q & tokenize(doc["text"]))
        if score:
            scored.append((score, doc))
    scored.sort(key=lambda item: item[0], reverse=True)
    return [doc for _, doc in scored[:top_k]]

def route_question(question: str, docs: list[dict]) -> str:
    if not docs:
        return "not_answered"
    if any("uncertain" in d["text"].lower() for d in docs):
        return "needs_review"
    return "answer"


## workflow dry run

먼저 API를 쓰지 않고 검색과 routing만 확인합니다.


In [ ]:
for question in [
    "What should API onboarding include?",
    "What is the private enterprise pricing policy?",
    "What happens with uncertain support cases?",
]:
    docs = retrieve(question)
    print("QUESTION:", question)
    print("DOCS:", [d["source_id"] for d in docs])
    print("ROUTE:", route_question(question, docs))
    print()


## API key 입력

각 실습 노트북은 독립적으로 실행됩니다. 아래 셀의 따옴표 안에 수업용 OpenAI API key를 붙여넣고 실행하세요.

- key가 들어간 노트북은 그대로 공유하지 않습니다.
- 제출하거나 화면 공유하기 전에는 key 문자열을 지웁니다.
- 이 자료에서는 `.env` 파일을 쓰지 않습니다. 학생이 열어야 할 파일 수를 줄이기 위해 노트북 안에서 직접 입력합니다.


In [ ]:
import os

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "여기에_수업용_API_KEY를_붙여넣으세요")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")

if not OPENAI_API_KEY or OPENAI_API_KEY.startswith("여기에_"):
    raise ValueError("환경변수 OPENAI_API_KEY를 설정하거나 이 셀의 자리에 수업용 API key를 붙여넣은 뒤 다시 실행하세요.")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("API key가 현재 노트북 실행 세션에 설정되었습니다.")


## 답변 단계 연결

검색과 routing 결과를 바탕으로 LLM에게 최종 JSON을 만들게 합니다.


In [ ]:
from openai import OpenAI

client = OpenAI()
question = "What should API onboarding include?"
docs = retrieve(question)
route = route_question(question, docs)
context = "\n\n".join(f"[{d['source_id']}] {d['text']}" for d in docs)

prompt = f"""
아래 workflow 상태를 바탕으로 답하세요.
route는 이미 결정되어 있으므로 그대로 사용하세요. 근거가 있으면 evidence의 source_id를 context의 ID로 적고,
quote는 원문에서 글자까지 정확히 복사하세요. answer route가 아니면 evidence는 빈 배열로 두세요.

route: {route}
question: {question}
context:
{context}
"""

response = client.responses.parse(
    model=OPENAI_MODEL,
    input=prompt,
    text_format=WorkflowResult,
    max_output_tokens=700,
    store=False,
)
result = response.output_parsed
if result is None:
    raise RuntimeError("구조화된 workflow 결과가 반환되지 않았습니다.")
if result.route != route:
    raise AssertionError(f"고정 route가 변경되었습니다: expected={route}, actual={result.route}")
print(result.model_dump_json(indent=2))
result


## 결과 확인

- workflow는 “모델에게 모든 것을 맡기는 것”이 아니라 단계를 나누는 방식입니다.
- 검색 결과가 없을 때 답을 지어내지 않는 branch가 있어야 합니다.
- 사람이 봐야 하는 애매한 case를 `needs_review`로 분리할 수 있습니다.
